#### Weighted Expert Ensemble Inference

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
# Import necessary libraries
import os
import sys
# Add the utils folder to the system path so we can import custom modules
sys.path.append(os.path.abspath(os.path.join("../..")))
import matplotlib.pyplot as plt
import torch
import numpy as np
import torch.nn as nn
import pandas as pd
from utils.reproducibility_utils import seed_everything
from utils.data_loadingbins import load_data,load_test_data

# Set the plotting style for better-looking visualizations
plt.style.available
plt.style.use('seaborn-v0_8')

# 1. Define the RNN Model
class RNN(nn.Module):
    def __init__(self, input_size, output_size, rnn_units, fc_units, dropout=[0.0, 0.0, 0.0]):
        super(RNN, self).__init__()
        
        self.input_dropout = nn.Dropout(dropout[0])
        self.rnn = nn.GRU(input_size, rnn_units, batch_first=True)
        self.rnn_dropout = nn.Dropout(dropout[1])
        self.fc = nn.Linear(rnn_units, fc_units)
        self.fc_dropout = nn.Dropout(dropout[2])
        self.output = nn.Linear(fc_units, output_size)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        x = self.input_dropout(x)
        rnn_out, _ = self.rnn(x)
        rnn_out = self.rnn_dropout(rnn_out[:, -1, :])  # Take the last time step
        fc_out = self.relu(self.fc(rnn_out))
        fc_out = self.fc_dropout(fc_out)
        out = self.output(fc_out)
        return out

# 2. Weighted Huber Loss Function
def weighted_huber_loss(predictions, targets, delta=1.0):
    # Standard Huber Loss
    loss = torch.nn.functional.huber_loss(predictions.squeeze(), targets, delta=delta, reduction='none')
    
    # Apply weights
    weights = torch.ones_like(targets)
    weights[targets.abs() > 2.3] = 4.0   # Medium weight (reduced)
    weights[targets.abs() > 5.0] = 10.0  # Few weight (reduced)
    
    return (weights * loss).mean()

# 3. Training Function
def train_one_epoch(model, dataloader, optimizer, device):
    model.train()
    running_loss = 0.0
    for x, y in dataloader:
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        preds = model(x)
        loss = nn.L1Loss() (preds.squeeze(), y)  # Using L1 Loss for training
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    return running_loss / len(dataloader)

# 4. Evaluation Function with Per-Band Analysis
def evaluate_model(model, dataloader, device):
    model.eval()
    results = []
    
    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            preds = model(x).squeeze()
            
            # Store predictions and targets for analysis
            res = pd.DataFrame({
                'target': y.cpu().numpy(),
                'pred': preds.cpu().numpy()
            })
            results.append(res)
            
    eval_df = pd.concat(results)
    eval_df['error_sq'] = (eval_df['target'] - eval_df['pred'])**2
    
    # Categorize into bands for reporting
    def get_band(val):
        if abs(val) > 5.0: return 'Few'
        if abs(val) > 2.3: return 'Medium'
        return 'Many'
    
    eval_df['band'] = eval_df['target'].apply(get_band)
    
    # Calculate Mean Squared Error per band
    band_metrics = eval_df.groupby('band')['error_sq'].mean().to_dict()
    total_mse = eval_df['error_sq'].mean()
    
    return total_mse, band_metrics

# --- MAIN EXECUTION ---
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 5. Load Data using our updated strategy
    # Note: Ensure load_data is imported from your dataloading.py
    train_loader, val_loader, pipeline = load_data(
        train_path='../../datasets_challenge/train',
        dist_csv_path='../../datasets_challenge/sideSlip_distribution.csv',
        val_file_name='train_part_020.csv', # Our "Drift Stress Test" file
        columns_to_drop = ["sideSlip", "vyCG", "time", "longitude","latitude"],
        sequence_length = 20,
        threshold = True,  # Enable velocity thresholding
        threshold_value = 3.0,  # Only keep samples where vehicle speed > 3 m/s
        threshold_column = "vxCG",  # Column to apply threshold on (longitudinal velocity)
        target='sideSlip',
        batchsize=64
    )

    # 6. Initialize Model
    # Set device for computation (CPU or GPU)
    device = "cpu"  # Change to "cuda" if you have a GPU available

    # Model hyperparameters
    input_size = 22  # Number of features (22 in this case)
    rnn_units = 48  # Number of units in RNN hidden state
    fc_units = 48  # Number of units in fully connected layer
    output_size = 1  # Single output (sideslip angle)
    lr = 0.0003  # Learning rate for optimizer

    # Initialize the model with specified architecture
    model = RNN(
        input_size=input_size, 
        output_size=output_size,
        rnn_units=rnn_units,
        fc_units=fc_units,
        dropout=[0.0, 0.0, 0.0]  # Dropout rates for [input, after_rnn, after_fc]
    ).to(device)

    # Define optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)  # Adam optimizer for weight updates

    # 7. Run Training Loop
    epochs = 20
    print(f"{'Epoch':<8} | {'Train Loss':<12} | {'Val MSE':<10} | {'Few-Shot MSE':<12}")
    print("-" * 60)

    for epoch in range(epochs):
        train_loss = train_one_epoch(model, train_loader, optimizer, device)
        val_mse, band_mse = evaluate_model(model, val_loader, device)
        
        few_mse = band_mse.get('Few', 0.0)
        
        print(f"{epoch+1:<8} | {train_loss:<12.5f} | {val_mse:<10.5f} | {few_mse:<12.5f}")

    print("\nFinal Performance on Validation Set (train_part_017.csv):")
    for band, mse in band_mse.items():
        print(f"Band: {band:<7} | MSE: {mse:.5f}")

VALIDATION: Loading train_part_020.csv
Calculating weights for WeightedRandomSampler...
Epoch    | Train Loss   | Val MSE    | Few-Shot MSE
------------------------------------------------------------
1        | 0.55298      | 0.06106    | 0.08152     
2        | 0.27433      | 0.04737    | 0.06307     
3        | 0.23956      | 0.03262    | 0.04738     
4        | 0.22286      | 0.03197    | 0.05223     
5        | 0.21088      | 0.03004    | 0.09102     
6        | 0.20172      | 0.03008    | 0.05293     
7        | 0.19494      | 0.02535    | 0.05559     
8        | 0.18717      | 0.02632    | 0.07912     
9        | 0.17362      | 0.02573    | 0.03625     
10       | 0.16504      | 0.02426    | 0.03746     
11       | 0.15930      | 0.02310    | 0.05834     
12       | 0.15545      | 0.02491    | 0.04312     
13       | 0.15107      | 0.02391    | 0.04820     
14       | 0.14685      | 0.02305    | 0.04725     
15       | 0.14396      | 0.02468    | 0.05124     
16       | 0.14031 

In [8]:
print("\nLoading test data...")
test_dataloader = load_test_data(test_path = "../../datasets_challenge/test",
                                 batchsize = 64,
                                 pipeline = pipeline, 
                                 load_anonymized_data=True)


Loading test data...
../../datasets_challenge/test\test_part_001.csv
../../datasets_challenge/test\test_part_002.csv
../../datasets_challenge/test\test_part_003.csv
../../datasets_challenge/test\test_part_004.csv
../../datasets_challenge/test\test_part_005.csv
../../datasets_challenge/test\test_part_006.csv
../../datasets_challenge/test\test_part_007.csv
../../datasets_challenge/test\test_part_008.csv
../../datasets_challenge/test\test_part_009.csv
../../datasets_challenge/test\test_part_010.csv
../../datasets_challenge/test\test_part_011.csv


In [9]:
# Generate predictions on test set
model.eval()
total_loss = 0  
test_preds = []
test_ground_truths = []  # Will contain placeholder values (all zeros)

with torch.no_grad():
    for step, (x, y) in enumerate(test_dataloader):
        X, y = x.to(device), y.to(device).reshape([len(y), 1])

        pred = model(X)
        loss = nn.L1Loss()(pred, y)
        total_loss += loss.item()
        # Note: loss is meaningless here since y contains placeholder values
        
        # Store predictions
        test_preds.append(pred.reshape(-1).cpu().numpy())
        test_ground_truths.append(y.reshape(-1).cpu().numpy())

    # Concatenate all batches
    test_ground_truths = np.concatenate(test_ground_truths)
    test_preds = np.concatenate(test_preds)
    
print("Shape of Predictions", test_preds.shape)

Shape of Predictions (49328,)


In [10]:
# Save predictions in the required format for submission
y_pred = pd.DataFrame({"predicted_sideslip": test_preds})
y_pred["id"] = range(len(test_preds))  # Add sample IDs (0 to 49327)

# Save to CSV file
y_pred.to_csv("predictionsvalidaation4.csv", index=False)

print(f"✓ Predictions saved to 'predictionsvalidaation4.csv'")
print(f"✓ Total samples: {len(test_preds)}")
print(f"✓ Ready for submission to the competition!")

✓ Predictions saved to 'predictionsvalidaation4.csv'
✓ Total samples: 49328
✓ Ready for submission to the competition!
